# HR-GNSS Earthquake Magnitude Estimation with Deep Learning

**Based on:** *Quinteros-Cartaya et al. (2024) — Exploring a CNN model for earthquake magnitude estimation using HR-GNSS data*, Journal of South American Earth Sciences 136, 104815.

---

This notebook reproduces the **real-data evaluation** from Section 5.3 of the paper (Figure 11), testing trained CNN models against real HR-GNSS displacement waveforms from six large earthquakes:

| Event | Region | Mw |
|---|---|---|
| Nicoya 2012 | Costa Rica | 7.6 |
| Mentawai 2010 | Indonesia | 7.7 |
| Iquique 2014 | Chile | 8.1 |
| Tehuantepec 2017 | Mexico | 8.2 |
| Illapel 2015 | Chile | 8.3 |
| Maule 2010 | Chile | 8.8 |

Two trained models are evaluated:
- **Case I** — 3 stations, Δ ≤ 3°, 181 s window
- **Case II** — 7 stations, Δ ≤ 3°, 181 s window

For each event, up to 500 random station combinations are drawn and evaluated. Results are plotted as violin + scatter overlays (coloured by median epicentral distance), matching Figure 11 of the paper.

---

## Prerequisites

```
pip install tensorflow obspy pandas matplotlib numpy
```

**Expected directory layout:**
```
dataset_root/
    Nicoya2012/
        disp/
            STATION.LXE.mseed
            STATION.LXN.mseed
            STATION.LXZ.mseed
        Nicoya2012_disp.chan
    Mentawai2010/
        ...
    ...

models_root/
    GNSS_M3S_181/
        model_Standard.h5
    GNSS_M7S_181/
        model_Standard.h5
```


## 1. Configuration

Set your local paths and evaluation parameters here. Everything else runs automatically.

In [ ]:
import os

# ── Path configuration ────────────────────────────────────────────────────────
# Root folder containing one sub-folder per earthquake event
DATA_ROOT = r"D:\dataset_ruhl_etal_2018"

# Root folder containing the two trained model sub-folders
MODEL_ROOT = r"D:\data science\project\Earthquake analysis with deep learning\clone2\DL-HRGNSS\trained_models"

# ── Event selection ───────────────────────────────────────────────────────────
# Remove events from this list if their data folders are unavailable.
EVENTS = [
    "Nicoya2012",
    "Mentawai2010",
    "Iquique2014",
    "Tehuantepec2017",
    "Illapel2015",
    "Maule2010",
]

# ── Evaluation hyperparameters ────────────────────────────────────────────────
# Max station combinations per event per case (paper uses 500)
MAX_COMBINATIONS = 500

# Station selection: only include stations within this epicentral distance (degrees).
# Set to None to use all available stations (matching paper's per-event approach).
MAX_RADIUS_DEG = None

# Normalisation applied to each station tensor before inference
NORMALIZE = "per_station_maxabs"

# Random seed (for reproducible station combination draws)
SEED = 42

# GPU batch size for model.predict()
PREDICTION_BATCH_SIZE = 128

# Output paths
OUTPUT_CSV  = "real_data_results.csv"
OUTPUT_BEST = "real_data_best_combinations.csv"
OUTPUT_FIG  = "real_data_figure11_reproduction.png"

# Derived model paths
MODEL_CASE_I  = os.path.join(MODEL_ROOT, "GNSS_M3S_181", "model_Standard.h5")
MODEL_CASE_II = os.path.join(MODEL_ROOT, "GNSS_M7S_181", "model_Standard.h5")

EVENT_FOLDERS = [os.path.join(DATA_ROOT, ev) for ev in EVENTS]

print("Configuration loaded.")
print(f"  Events   : {EVENTS}")
print(f"  Model I  : {MODEL_CASE_I}")
print(f"  Model II : {MODEL_CASE_II}")

## 2. Library Imports

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"   # suppress TF C++ warnings

import re
import math
import random
import warnings
from dataclasses import dataclass
from itertools import combinations
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from obspy import read
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from tensorflow.keras.constraints import max_norm

warnings.filterwarnings("ignore", category=FutureWarning)
print("Libraries imported successfully.")

## 3. Constants and Known Event Metadata

These match the values used in the paper for the six real-data test events.

In [ ]:
# ── Model input shape constants ───────────────────────────────────────────────
NT_DEFAULT = 181      # time steps (seconds after origin time, including t=0)
NC = 3                # channels: U (up/LXZ), N (north/LXN), E (east/LXE)
CHANNEL_ORDER = ("U", "N", "E")

# ── Known event properties (from the paper, Table 1 and Section 5.3) ─────────
KNOWN_EVENT_COORDS = {
    "Nicoya2012":      {"lat":  10.085, "lon":  -85.315, "depth_km": 20.0, "magnitude": 7.6},
    "Mentawai2010":    {"lat":  -3.490, "lon":  100.080, "depth_km": 20.0, "magnitude": 7.7},
    "Iquique2014":     {"lat": -19.610, "lon":  -70.770, "depth_km": 25.0, "magnitude": 8.1},
    "Tehuantepec2017": {"lat":  14.760, "lon":  -94.100, "depth_km": 47.0, "magnitude": 8.2},
    "Illapel2015":     {"lat": -31.640, "lon":  -71.740, "depth_km": 22.4, "magnitude": 8.3},
    "Maule2010":       {"lat": -35.910, "lon":  -72.730, "depth_km": 35.0, "magnitude": 8.8},
}

# Ordered from lowest to highest Mw — matches the paper's x-axis in Figure 11
EVENT_ORDER_BY_MW = [
    "Nicoya2012",
    "Mentawai2010",
    "Iquique2014",
    "Tehuantepec2017",
    "Illapel2015",
    "Maule2010",
]

print("Event metadata:")
for name, info in KNOWN_EVENT_COORDS.items():
    print(f"  {name:<20} Mw {info['magnitude']}  depth {info['depth_km']} km")

## 4. Data Classes

In [ ]:
@dataclass
class StationMeta:
    """Metadata for one HR-GNSS station, parsed from the .chan file."""
    code: str
    net: Optional[str] = None
    loc: Optional[str] = None
    lat: Optional[float] = None
    lon: Optional[float] = None
    elev: Optional[float] = None
    samplerate: Optional[float] = None
    gain: Optional[float] = None
    units: Optional[str] = None


@dataclass
class EventMeta:
    """Metadata for one earthquake event."""
    event_id: Optional[str] = None
    origin_time: Optional[str] = None
    latitude: Optional[float] = None
    longitude: Optional[float] = None
    depth_km: Optional[float] = None
    magnitude: Optional[float] = None

print("Data classes defined.")

## 5. Geometry Utilities

In [ ]:
def haversine_km(lat1: float, lon1: float, lat2: float, lon2: float) -> float:
    """Great-circle distance between two points (degrees → km)."""
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp, dl = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def km_to_deg(dist_km: float) -> float:
    """Approximate conversion from km to degrees (1° ≈ 111 km)."""
    return dist_km / 111.0


def azimuth_deg(
    event_lat: float, event_lon: float,
    station_lat: float, station_lon: float,
) -> float:
    """Forward azimuth from event to station (degrees, 0–360)."""
    lat1 = math.radians(event_lat)
    lat2 = math.radians(station_lat)
    dlon = math.radians(station_lon - event_lon)
    x = math.sin(dlon) * math.cos(lat2)
    y = math.cos(lat1) * math.sin(lat2) - math.sin(lat1) * math.cos(lat2) * math.cos(dlon)
    return math.degrees(math.atan2(x, y)) % 360.0

print("Geometry utilities defined.")

## 6. File Discovery and MiniSEED Reading

In [ ]:
def get_disp_folder(event_folder: str) -> str:
    """Return the 'disp/' sub-folder, or raise if absent."""
    p = os.path.join(event_folder, "disp")
    if not os.path.isdir(p):
        raise FileNotFoundError(f"'disp' folder not found inside: {event_folder}")
    return p


def find_chan_file(event_folder: str) -> str:
    """Locate the .chan metadata file inside the event folder."""
    candidates = [f for f in os.listdir(event_folder) if f.lower().endswith(".chan")]
    if not candidates:
        raise FileNotFoundError(f"No .chan file found in: {event_folder}")
    preferred = [f for f in candidates if f.lower().endswith("_disp.chan")]
    return os.path.join(event_folder, preferred[0] if preferred else candidates[0])


def discover_station_components(
    disp_folder: str,
) -> Dict[str, Dict[str, str]]:
    """
    Scan a 'disp/' folder for MiniSEED files matching STATION.{LXE|LXN|LXZ}.mseed.
    Returns a dict mapping station code → {component: path}.
    Only stations that have all three components are returned.
    """
    pattern = re.compile(r"^([^.]+)\.(LXE|LXN|LXZ)\.mseed$", re.IGNORECASE)
    station_files: Dict[str, Dict[str, str]] = {}
    for fname in os.listdir(disp_folder):
        m = pattern.match(fname)
        if not m:
            continue
        sta = m.group(1).upper()
        comp = m.group(2).upper()
        station_files.setdefault(sta, {})[comp] = os.path.join(disp_folder, fname)
    return {
        sta: comps
        for sta, comps in station_files.items()
        if {"LXE", "LXN", "LXZ"}.issubset(comps)
    }


def read_trace(path: str) -> np.ndarray:
    """Read the first trace from a MiniSEED file and return it as float32."""
    stream = read(path)
    if not stream:
        raise ValueError(f"No traces in: {path}")
    return np.asarray(stream[0].data, dtype=np.float32)


def enforce_length(arr: np.ndarray, nt: int) -> np.ndarray:
    """
    Crop or zero-pad a 1-D array to exactly `nt` samples.
    Short traces (real data can arrive late or be clipped) are zero-padded
    at the end, consistent with how the paper handles edge cases.
    """
    if len(arr) >= nt:
        return arr[:nt].astype(np.float32)
    out = np.zeros(nt, dtype=np.float32)
    out[:len(arr)] = arr
    return out


def load_station_tensor(
    disp_folder: str,
    station_code: str,
    nt: int = NT_DEFAULT,
    normalize: Optional[str] = "per_station_maxabs",
) -> np.ndarray:
    """
    Load all three components for one station and return an (nt, 3) tensor.
    Channel order is (U=LXZ, N=LXN, E=LXE), matching Fig. 1 of the paper.

    normalize options
    -----------------
    'per_station_maxabs' : divide by max(|x|) across all channels & samples.
    'per_channel_std'    : z-score each channel independently.
    None                 : no normalisation (raw displacement in metres).
    """
    sta = station_code.upper()
    u = enforce_length(read_trace(os.path.join(disp_folder, f"{sta}.LXZ.mseed")), nt)
    n = enforce_length(read_trace(os.path.join(disp_folder, f"{sta}.LXN.mseed")), nt)
    e = enforce_length(read_trace(os.path.join(disp_folder, f"{sta}.LXE.mseed")), nt)
    tensor = np.stack([u, n, e], axis=-1)          # shape (nt, 3)

    if normalize == "per_station_maxabs":
        denom = np.max(np.abs(tensor))
        if denom > 0:
            tensor = tensor / denom
    elif normalize == "per_channel_std":
        mu  = tensor.mean(axis=0, keepdims=True)
        sig = tensor.std(axis=0,  keepdims=True)
        sig[sig == 0] = 1.0
        tensor = (tensor - mu) / sig
    elif normalize is not None:
        raise ValueError(f"Unknown normalize option: {normalize!r}")

    return tensor.astype(np.float32)

print("File I/O helpers defined.")

## 7. Station Metadata Parsing (.chan Files)

In [ ]:
def parse_chan_file(
    chan_path: str,
    valid_station_codes: set,
) -> Dict[str, StationMeta]:
    """
    Parse a .chan file and extract station coordinates.

    Expected line format (space-delimited):
        NET  STA  LOC  CHAN  LAT  LON  ELEV  SAMPLERATE  GAIN  UNITS  ...

    Only stations present in `valid_station_codes` (those with waveform data)
    are returned. When the same station appears on multiple lines (one per
    channel), only the first occurrence is stored.
    """
    meta: Dict[str, StationMeta] = {}
    with open(chan_path, encoding="utf-8", errors="ignore") as fh:
        for raw in fh:
            line = raw.strip()
            if not line or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 10:
                continue
            net, sta, loc, chan = parts[0], parts[1].upper(), parts[2], parts[3].upper()
            if sta not in valid_station_codes or sta in meta:
                continue
            meta[sta] = StationMeta(
                code=sta, net=net, loc=loc,
                lat=float(parts[4]), lon=float(parts[5]), elev=float(parts[6]),
                samplerate=float(parts[7]), gain=float(parts[8]), units=parts[9],
            )
    return meta

print("Chan file parser defined.")

## 8. CNN Model Architecture and Loading

This replicates the exact architecture from Figure 2 / Section 2 of the paper:
six 2D-conv layers, three max-pooling layers, three dense layers (128 → 32 → 1).

In [ ]:
def build_cnn(
    nst: int,
    nt: int,
    nc: int = NC,
) -> keras.Model:
    """
    Build the CNN architecture from the paper (Fig. 2).

    Input shape: (nst, nt, nc) = (stations, time_steps, channels)
    Output: scalar earthquake moment magnitude Mw.
    """
    model = Sequential(name=f"GNSS_CNN_{nst}S_{nt}t")
    # Block 1: 12 filters, no padding (valid)
    model.add(Conv2D(12,  (1, 3), activation="relu", input_shape=(nst, nt, nc)))
    model.add(MaxPooling2D((1, 2)))
    # Block 2: 24 → 32 filters, same padding
    model.add(Conv2D(24,  (1, 3), activation="relu", padding="same"))
    model.add(Conv2D(32,  (1, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((1, 2)))
    # Block 3: 64 → 128 filters, same padding
    model.add(Conv2D(64,  (1, 3), activation="relu", padding="same"))
    model.add(Conv2D(128, (1, 3), activation="relu", padding="same"))
    model.add(MaxPooling2D((1, 2)))
    # Block 4: 256 filters, no padding (valid)
    model.add(Conv2D(256, (1, 3), activation="relu"))
    # Dense head
    model.add(Flatten())
    model.add(Dense(128, activation="relu", kernel_constraint=max_norm(3)))
    model.add(Dense(32,  activation="relu", kernel_constraint=max_norm(3)))
    model.add(Dense(1,   activation="linear"))
    return model


def load_model(
    model_path: str,
    nst: int,
    nt: int,
) -> keras.Model:
    """
    Load a trained model from an .h5 file.  Falls back to checkpoint weights
    (cp_Standard.ckpt.*) if the full-model load fails.
    """
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model file not found: {model_path}")

    try:
        model = keras.models.load_model(model_path, compile=False)
        print(f"  Loaded full model: {model_path}")
        return model
    except Exception as exc:
        print(f"  Full-model load failed ({exc}). Trying checkpoint fallback …")

    model_dir = os.path.dirname(model_path)
    ckpt_prefix = os.path.join(model_dir, "cp_Standard.ckpt")
    ckpt_index  = ckpt_prefix + ".index"
    if not os.path.exists(ckpt_index):
        raise FileNotFoundError(
            f"Checkpoint not found either. Expected: {ckpt_index}"
        )
    model = build_cnn(nst=nst, nt=nt)
    model.load_weights(ckpt_prefix)
    print(f"  Loaded weights from checkpoint: {ckpt_prefix}")
    return model

print("Model helpers defined.")

## 9. Station Combination Engine

For each event we:
1. Filter stations to those with coordinates and (optionally) within `max_radius_deg`.
2. Enumerate all possible combinations of `nst` stations.
3. Randomly shuffle and cap at `MAX_COMBINATIONS`, matching the paper's 500-combination protocol.

In [ ]:
def usable_stations(
    station_files: Dict[str, Dict[str, str]],
    station_meta: Dict[str, StationMeta],
    event_meta: EventMeta,
    max_radius_deg: Optional[float],
) -> List[str]:
    """
    Return alphabetically sorted station codes that
      - have waveform data (all three components), AND
      - have coordinate metadata in the .chan file, AND
      - (optionally) lie within max_radius_deg of the epicentre.
    """
    usable = []
    for code in sorted(station_files):
        meta = station_meta.get(code)
        if meta is None or meta.lat is None or meta.lon is None:
            continue
        if max_radius_deg is not None:
            dist_km  = haversine_km(event_meta.latitude, event_meta.longitude,
                                    meta.lat, meta.lon)
            if km_to_deg(dist_km) > max_radius_deg:
                continue
        usable.append(code)
    return usable


def draw_combinations(
    station_codes: List[str],
    nst: int,
    seed: int = 42,
    max_combinations: Optional[int] = None,
) -> List[Tuple[str, ...]]:
    """
    Generate all C(n, nst) combinations, shuffle them, then cap at
    `max_combinations`. Returns the shuffled (possibly truncated) list.
    """
    if len(station_codes) < nst:
        raise ValueError(
            f"Only {len(station_codes)} usable stations but need {nst}."
        )
    rng   = random.Random(seed)
    combos = list(combinations(sorted(station_codes), nst))
    rng.shuffle(combos)
    if max_combinations is not None:
        combos = combos[:max_combinations]
    return combos


def combo_distance_stats(
    combo: Tuple[str, ...],
    station_meta: Dict[str, StationMeta],
    event_meta: EventMeta,
) -> Dict[str, float]:
    """
    Compute epicentral distance statistics for a station combination.
    These are used for colouring the scatter plot (Fig. 11).
    """
    dists_km, dists_deg, azims = [], [], []
    for code in combo:
        m = station_meta[code]
        dkm = haversine_km(event_meta.latitude, event_meta.longitude, m.lat, m.lon)
        dists_km.append(dkm)
        dists_deg.append(km_to_deg(dkm))
        azims.append(azimuth_deg(event_meta.latitude, event_meta.longitude, m.lat, m.lon))
    return {
        "median_distance_deg": float(np.median(dists_deg)),
        "median_distance_km":  float(np.median(dists_km)),
        "min_distance_deg":    float(np.min(dists_deg)),
        "max_distance_deg":    float(np.max(dists_deg)),
        "median_azimuth_deg":  float(np.median(azims)),
    }

print("Station combination helpers defined.")

## 10. Core Evaluation Function

This is the heart of the notebook. For each (event, case) pair it:
1. Loads waveform tensors for all usable stations.
2. Draws random station combinations.
3. Runs batched model inference.
4. Computes the prediction error versus the catalogue magnitude.

In [ ]:
def evaluate_event(
    event_folder: str,
    model_path: str,
    nst: int,
    case_label: str,
    nt: int = NT_DEFAULT,
    normalize: Optional[str] = "per_station_maxabs",
    seed: int = 42,
    max_radius_deg: Optional[float] = None,
    max_combinations: Optional[int] = 500,
    batch_size: int = 128,
) -> pd.DataFrame:
    """
    Evaluate one event × one case.

    Returns a DataFrame with one row per station combination tested,
    containing the prediction error and distance statistics.
    """
    # ── Event metadata ────────────────────────────────────────────────────────
    event_name = os.path.basename(os.path.normpath(event_folder))
    if event_name not in KNOWN_EVENT_COORDS:
        raise ValueError(f"Event '{event_name}' not found in KNOWN_EVENT_COORDS.")
    info = KNOWN_EVENT_COORDS[event_name]
    event_meta = EventMeta(
        event_id=event_name,
        latitude=info["lat"],
        longitude=info["lon"],
        depth_km=info["depth_km"],
        magnitude=info["magnitude"],
    )

    # ── Discover waveform files and parse station metadata ────────────────────
    disp_folder  = get_disp_folder(event_folder)
    chan_path    = find_chan_file(event_folder)
    station_files = discover_station_components(disp_folder)
    station_meta  = parse_chan_file(chan_path, set(station_files.keys()))

    # ── Filter stations and draw combinations ─────────────────────────────────
    codes  = usable_stations(station_files, station_meta, event_meta, max_radius_deg)
    combos = draw_combinations(codes, nst, seed=seed, max_combinations=max_combinations)
    print(f"  {event_name} | {case_label}: {len(codes)} stations → {len(combos)} combinations")

    # ── Pre-load all needed station tensors into memory ───────────────────────
    tensor_cache: Dict[str, np.ndarray] = {
        code: load_station_tensor(disp_folder, code, nt=nt, normalize=normalize)
        for code in codes
    }

    # ── Load trained model ────────────────────────────────────────────────────
    model = load_model(model_path, nst=nst, nt=nt)

    # ── Batched inference ─────────────────────────────────────────────────────
    rows = []
    for start in range(0, len(combos), batch_size):
        chunk = combos[start : start + batch_size]

        # Stack tensors into batch: shape (batch, nst, nt, 3)
        x_batch = np.stack(
            [np.stack([tensor_cache[st] for st in combo], axis=0) for combo in chunk],
            axis=0,
        ).astype(np.float32)

        preds = model.predict(x_batch, verbose=0).reshape(-1)
        # Paper rounds predictions to 1 decimal place
        preds = np.round(preds.astype(float), 1)

        for combo, pred in zip(chunk, preds):
            dist_stats = combo_distance_stats(combo, station_meta, event_meta)
            error = float(pred - event_meta.magnitude)
            rows.append({
                "case":              case_label,
                "nst":               nst,
                "nt":                nt,
                "event_id":          event_name,
                "true_mw":           float(event_meta.magnitude),
                "pred_mw":           float(pred),
                "error":             error,          # pred − true  (positive = overestimate)
                "abs_error":         abs(error),
                "stations":          ",".join(combo),
                "n_usable_stations": len(codes),
                "n_combinations":    len(combos),
                **dist_stats,
            })

    return pd.DataFrame(rows)


def evaluate_all_events(
    event_folders: List[str],
    model_path_case_i:  str,
    model_path_case_ii: str,
    normalize: Optional[str] = "per_station_maxabs",
    seed: int = 42,
    max_radius_deg: Optional[float] = None,
    max_combinations: Optional[int] = 500,
    batch_size: int = 128,
) -> pd.DataFrame:
    """
    Run Case I and Case II evaluation across all supplied event folders.
    Returns a combined DataFrame.
    """
    all_dfs = []
    cases = [
        ("Case I (3 sta, 181 s)",  3, model_path_case_i),
        ("Case II (7 sta, 181 s)", 7, model_path_case_ii),
    ]
    for label, nst, model_path in cases:
        print(f"\n=== {label} ===")
        for event_folder in event_folders:
            try:
                df = evaluate_event(
                    event_folder=event_folder,
                    model_path=model_path,
                    nst=nst,
                    case_label=label,
                    nt=NT_DEFAULT,
                    normalize=normalize,
                    seed=seed,
                    max_radius_deg=max_radius_deg,
                    max_combinations=max_combinations,
                    batch_size=batch_size,
                )
                all_dfs.append(df)
            except Exception as exc:
                print(f"  !! Skipping {os.path.basename(event_folder)}: {exc}")

    return pd.concat(all_dfs, ignore_index=True)

print("Evaluation functions defined.")

## 11. Run the Evaluation

This cell performs inference and may take several minutes depending on your hardware.

In [ ]:
print("Starting evaluation …\n")

results_df = evaluate_all_events(
    event_folders=EVENT_FOLDERS,
    model_path_case_i=MODEL_CASE_I,
    model_path_case_ii=MODEL_CASE_II,
    normalize=NORMALIZE,
    seed=SEED,
    max_radius_deg=MAX_RADIUS_DEG,
    max_combinations=MAX_COMBINATIONS,
    batch_size=PREDICTION_BATCH_SIZE,
)

results_df.to_csv(OUTPUT_CSV, index=False)
print(f"\nResults saved → {OUTPUT_CSV}")
print(f"Total predictions: {len(results_df):,}")
results_df.head()

## 12. Summary Statistics

Reproduces the RMS values shown beneath each column in Figure 11 of the paper.

In [ ]:
def rms(errors: Iterable[float]) -> float:
    """Root-mean-squared error."""
    arr = np.asarray(list(errors), dtype=float)
    return float(np.sqrt(np.mean(arr ** 2))) if len(arr) else np.nan


def pct_within(errors: Iterable[float], threshold: float = 0.5) -> float:
    """Fraction of |error| ≤ threshold, expressed as a percentage."""
    arr = np.asarray(list(errors), dtype=float)
    return float(np.mean(np.abs(arr) <= threshold) * 100) if len(arr) else np.nan


summary = (
    results_df
    .groupby(["case", "event_id", "true_mw"], as_index=False)
    .agg(
        n_combinations=("error", "size"),
        mean_error=("error", "mean"),
        median_error=("error", "median"),
        std_error=("error", "std"),
        rms_error=("error", rms),
        mae=("abs_error", "mean"),
        min_error=("error", "min"),
        max_error=("error", "max"),
        pct_within_0_5=("error", lambda x: pct_within(x, 0.5)),
        median_dist_deg=("median_distance_deg", "median"),
    )
)

# Display in event-magnitude order
event_rank = {e: i for i, e in enumerate(EVENT_ORDER_BY_MW)}
summary["_rank"] = summary["event_id"].map(event_rank).fillna(99)
summary = summary.sort_values(["case", "_rank"]).drop(columns="_rank")

print("Summary statistics (one row per event × case):")
print(summary.to_string(index=False))

## 13. Best Combinations per Event

Identifies the single combination closest to the true catalogue magnitude for each event × case pair.

In [ ]:
best_idx = results_df.groupby(["case", "event_id"])["abs_error"].idxmin()
best_df  = results_df.loc[best_idx, [
    "case", "event_id", "true_mw", "pred_mw",
    "error", "stations", "median_distance_deg",
]].copy()

best_df.to_csv(OUTPUT_BEST, index=False)
print(f"Best combinations saved → {OUTPUT_BEST}\n")
print(best_df.to_string(index=False))

## 14. Publication-Quality Figure

Reproduces Figure 11 from the paper:
- **Top panel (a):** Case I — 3 stations, 181 s.
- **Bottom panel (b):** Case II — 7 stations, 181 s.
- Violin plots show the full error distribution per event.
- Scatter points are coloured by the median epicentral distance of the combination.
- RMS values are printed below each event column.

In [ ]:
def plot_figure11(
    results_df: pd.DataFrame,
    event_order: List[str],
    output_path: Optional[str] = None,
    cmap: str = "viridis",
) -> None:
    """
    Reproduce Figure 11 of Quinteros-Cartaya et al. (2024).

    Two vertically stacked subplots (a) Case I, (b) Case II.
    Each subplot shows:
      • violin plot of the error distribution per earthquake
      • scatter overlay coloured by median epicentral distance of the combination
      • horizontal dashed line at zero error
      • RMS value annotated below each event
    """
    # ── Publication style ─────────────────────────────────────────────────────
    mpl.rcParams.update({
        "font.family":      "serif",
        "font.size":        12,
        "axes.labelsize":   13,
        "axes.titlesize":   14,
        "xtick.labelsize":  11,
        "ytick.labelsize":  11,
        "legend.fontsize":  11,
        "figure.dpi":       150,
    })

    # ── Filter to only events present in the results ───────────────────────────
    present_events = results_df["event_id"].unique()
    event_order    = [e for e in event_order if e in present_events]
    n_events       = len(event_order)

    # ── Case order (matches paper: Case I top, Case II bottom) ─────────────────
    case_order = sorted(results_df["case"].unique())
    n_cases    = len(case_order)

    # ── Shared colour scale: median epicentral distance (degrees) ──────────────
    vmin = float(results_df["median_distance_deg"].min())
    vmax = float(results_df["median_distance_deg"].max())
    norm = mpl.colors.Normalize(vmin=vmin, vmax=vmax)
    cm   = mpl.cm.get_cmap(cmap)

    rng = np.random.default_rng(42)   # reproducible jitter

    # ── Figure layout ─────────────────────────────────────────────────────────
    fig, axes = plt.subplots(
        n_cases, 1,
        figsize=(10, 3.8 * n_cases),
        sharex=True,
        sharey=True,
        constrained_layout=False,
    )
    if n_cases == 1:
        axes = [axes]

    last_scatter = None
    Y_LIM = (-1.05, 1.30)   # matches paper's y-axis range
    RMS_Y = -0.90            # fixed y-position for RMS annotation

    for ax_idx, (ax, case) in enumerate(zip(axes, case_order)):
        case_df = results_df[results_df["case"] == case]

        # ── Violin plots ──────────────────────────────────────────────────────
        violin_data, violin_pos = [], []
        for i, eid in enumerate(event_order):
            vals = case_df.loc[case_df["event_id"] == eid, "error"].dropna().values
            if len(vals) > 1:
                violin_data.append(vals)
                violin_pos.append(i)

        if violin_data:
            vp = ax.violinplot(
                violin_data,
                positions=violin_pos,
                widths=0.70,
                showmeans=False,
                showmedians=True,
                showextrema=True,
            )
            # Style: light yellow fill with orange edges (matches Fig. 11 palette)
            for body in vp["bodies"]:
                body.set_facecolor("#FFF2CC")
                body.set_edgecolor("#F6B26B")
                body.set_alpha(0.65)
                body.set_linewidth(1.0)
            for key in ["cbars", "cmins", "cmaxes", "cmedians"]:
                if key in vp:
                    vp[key].set_color("#CC2255")
                    vp[key].set_linewidth(1.5)

        # ── Scatter overlay (coloured by median epicentral distance) ──────────
        for i, eid in enumerate(event_order):
            sub = case_df[case_df["event_id"] == eid]
            if sub.empty:
                continue
            n_pts  = len(sub)
            jitter = rng.uniform(-0.14, 0.14, size=n_pts)
            pt_sz  = 18 if n_pts > 300 else 30
            last_scatter = ax.scatter(
                np.full(n_pts, i) + jitter,
                sub["error"].values,
                c=sub["median_distance_deg"].values,
                cmap=cmap, norm=norm,
                s=pt_sz, alpha=0.70,
                edgecolors="k", linewidths=0.3,
                zorder=3,
            )

        # ── Zero-error reference line (dashed, as in paper) ───────────────────
        ax.axhline(0, linestyle="--", linewidth=1.2, color="#AAAAAA", alpha=0.85, zorder=1)

        # ── RMS annotations (below each event column) ─────────────────────────
        for i, eid in enumerate(event_order):
            sub = case_df[case_df["event_id"] == eid]
            if sub.empty:
                continue
            rms_val = rms(sub["error"])
            ax.text(
                i, RMS_Y, f"{rms_val:.2f}",
                ha="center", va="center",
                fontsize=10, fontweight="bold", color="#222222",
            )

        # ── "RMS=" label at left margin ───────────────────────────────────────
        ax.text(
            0.01, (RMS_Y - Y_LIM[0]) / (Y_LIM[1] - Y_LIM[0]),
            "RMS=",
            transform=ax.transAxes,
            ha="left", va="center",
            fontsize=10, fontweight="bold", color="#222222",
        )

        # ── Case label box ────────────────────────────────────────────────────
        ax.text(
            0.50, 0.91, case,
            transform=ax.transAxes,
            ha="center", va="center", fontsize=12,
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                      edgecolor="#CCCCCC", alpha=0.92, linewidth=1.0),
            zorder=5,
        )

        # ── Subplot letter label ──────────────────────────────────────────────
        ax.text(
            -0.11, 0.96, f"({chr(97 + ax_idx)})",
            transform=ax.transAxes,
            ha="right", va="top",
            fontsize=14, fontweight="bold",
        )

        # ── Axis formatting ───────────────────────────────────────────────────
        ax.set_ylabel("Error of\nMagnitude Estimation", fontsize=12)
        ax.set_ylim(*Y_LIM)
        ax.set_xlim(-0.6, n_events - 0.4)
        ax.grid(axis="x", alpha=0.10, linestyle="-", linewidth=0.5)
        ax.yaxis.grid(True, alpha=0.15, linestyle=":")

    # ── X-axis tick labels (event name + year + Mw on separate lines) ─────────
    xlabels = []
    for eid in event_order:
        mw  = KNOWN_EVENT_COORDS.get(eid, {}).get("magnitude", np.nan)
        m   = re.match(r"([A-Za-z]+)(\d{4})", eid)
        lbl = f"{m.group(1)}\n{m.group(2)}\nMw {mw:.1f}" if m else eid
        xlabels.append(lbl)

    axes[-1].set_xticks(range(n_events))
    axes[-1].set_xticklabels(xlabels, fontsize=10)

    # ── Main title ────────────────────────────────────────────────────────────
    fig.suptitle(
        "Real Data — Magnitude Estimation Errors\n"
        "(Quinteros-Cartaya et al. 2024, Fig. 11 reproduction)",
        fontsize=13, fontweight="bold", y=0.99,
    )

    # ── Shared colour bar ─────────────────────────────────────────────────────
    if last_scatter is not None:
        cbar = fig.colorbar(
            last_scatter, ax=axes,
            fraction=0.025, pad=0.03, aspect=30,
        )
        cbar.set_label("Median Epicentral Distance (Δ°)", fontsize=11)
        cbar.ax.tick_params(labelsize=9)

    fig.subplots_adjust(left=0.13, right=0.87, top=0.94, bottom=0.10, hspace=0.05)

    if output_path:
        fig.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
        print(f"Figure saved → {output_path}")

    plt.show()

print("Plotting function defined.")

## 15. Generate the Figure

In [ ]:
plot_figure11(
    results_df=results_df,
    event_order=EVENT_ORDER_BY_MW,
    output_path=OUTPUT_FIG,
    cmap="viridis",
)

## 16. Optional: Run on Modal Cloud (GPU)

If local inference is too slow (e.g., evaluating all events with all combinations), you can offload to [Modal](https://modal.com). Install with `pip install modal` and authenticate with `modal setup`.

The cell below is **not executed automatically**; change `RUN_ON_MODAL = True` to use it.

In [ ]:
RUN_ON_MODAL = False  # ← set True to run on Modal cloud GPU

if RUN_ON_MODAL:
    import modal
    import io

    modal_app = modal.App("gnss-magnitude-inference")

    modal_image = (
        modal.Image.debian_slim(python_version="3.11")
        .pip_install("tensorflow==2.15.0", "obspy", "pandas", "matplotlib", "numpy")
        .add_local_dir(DATA_ROOT,  remote_path="/dataset")
        .add_local_dir(MODEL_ROOT, remote_path="/models")
    )

    @modal_app.function(image=modal_image, gpu="any", timeout=86400)
    def run_remote() -> str:
        import os, json
        # Inline the evaluation logic so it runs in the cloud container
        # (all helpers defined above need to be importable; consider saving
        #  them to a separate .py file and add_local_file()ing it instead)
        raise NotImplementedError(
            "Extract the pipeline functions to gnss_pipeline.py and import them here."
        )

    with modal_app.run():
        json_str = run_remote.remote()
    results_df = pd.read_json(io.StringIO(json_str), orient="records")
    results_df.to_csv(OUTPUT_CSV, index=False)
    print(f"Cloud results saved → {OUTPUT_CSV}")
else:
    print("Modal cloud execution skipped (RUN_ON_MODAL = False).")

---

## Interpretation Notes

| Finding | Paper Section |
|---|---|
| Errors are generally low for Mw 8.1–8.8 events (Iquique, Tehuantepec, Illapel, Maule) | §5.3 |
| Nicoya (Mw 7.6) shows the highest RMS (~0.49 in Case I) because all stations cluster within 1° and exhibit large near-source displacements rarely seen during training | §5.3 point 1 |
| Mentawai Case III (not implemented here) shows elevated error because only one station is at Δ > 3° | §5.3 point 2 |
| Scatter coloured by median epicentral distance shows that combinations with small median distances (dark) tend to cluster more tightly around zero error | Fig. 11 colour bar |
| Case II (7 stations) generally has lower RMS than Case I (3 stations) for the largest events | §5.3 |

**Reference:** Quinteros-Cartaya C., Köhler J., Li W., Faber J., Srivastava N. (2024). *Exploring a CNN model for earthquake magnitude estimation using HR-GNSS data.* Journal of South American Earth Sciences 136, 104815. https://doi.org/10.1016/j.jsames.2024.104815